In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the processed customer dataset
df = pd.read_parquet("../output/perfectpetclub_churn_features.parquet")

# Check the dataset
print("Total customers:", len(df))
print("Total columns:", len(df.columns))
print("Columns:", df.columns.tolist())

df.head()

## Customer Retention Overview

This chart shows the percentage of customers who have enabled or disabled subscription auto renewal. Customers who have disabled auto renewal are classified as churned for this analysis.

In [1]:


churn_counts = df["is_churned"].value_counts().reindex([0, 1], fill_value=0)
total = churn_counts.sum()

fig, ax = plt.subplots(figsize=(9, 5))

colors = ["#25547C", "#E99B3C"]

wedges, _ = ax.pie(
    churn_counts,
    startangle=90,
    counterclock=False,
    colors=colors,
    wedgeprops={"width": 0.35, "edgecolor": "white", "linewidth": 2}
)


ax.text(
    0, 0, f"{total}\nCustomers",
    ha="center",
    va="center",
    fontsize=16,
    fontweight="bold"
)


labels = [
    f"Auto-renew On: {churn_counts[0]} ({churn_counts[0]/total:.1%})",
    f"Auto-renew Off: {churn_counts[1]} ({churn_counts[1]/total:.1%})"
]

ax.legend(
    wedges,
    labels,
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False,
    fontsize=11
)

ax.set_title(
    "PerfectPetClub | Customer Retention Overview",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout()


fig.savefig(
    "../output/customer_churn_overview.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

NameError: name 'df' is not defined

## Churn Rate by Subscription Tier

The chart below compares the proportion of customers who have disabled auto renewal across the three subscription tiers (Basic, Premium, and SuperSaver), revealing differences in churn by plan type.

In [ ]:

tier_summary = df.groupby("subscription_tier")["is_churned"].agg(
    churn_rate="mean",
    customers="count"
).reset_index()

tier_summary["churn_rate"] = tier_summary["churn_rate"] * 100
tier_summary = tier_summary.sort_values("churn_rate", ascending=False)


tier_colors = {
    "Basic": "#2D6A73",       
    "Premium": "#B56576",     
    "SuperSaver": "#D9A441"   
}

colors = [tier_colors.get(t, "#666666") for t in tier_summary["subscription_tier"]]

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.barh(
    tier_summary["subscription_tier"],
    tier_summary["churn_rate"],
    color=colors,
    height=0.6,
    edgecolor="white",
    linewidth=1.5
)


ax.invert_yaxis()


for bar, rate, count in zip(
    bars,
    tier_summary["churn_rate"],
    tier_summary["customers"]
):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.1f}% ({count} customers)",
        va="center",
        fontsize=10,
        fontweight="bold",
        color="#333333"
    )

ax.set_xlim(0, max(55, tier_summary["churn_rate"].max() + 12))
ax.set_xlabel("Churn rate (% customers with auto-renew disabled)", fontsize=11)
ax.set_ylabel("Subscription tier", fontsize=11)
ax.set_title(
    "PerfectPetClub | Churn Rate by Subscription Tier",
    fontsize=14,
    fontweight="bold",
    pad=15,
    loc="left"
)

ax.xaxis.grid(True, alpha=0.25, linestyle="--")
ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#999999")
ax.spines["bottom"].set_color("#999999")

plt.tight_layout()
plt.savefig("../output/chart_churn_by_tier.png", dpi=150, bbox_inches="tight")
plt.show()